In [ ]:
import pickle
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import matthews_corrcoef
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')


In [ ]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


# ── Model (best architecture from 0.7207 run) ─────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, ch//8), nn.ReLU(),
                                nn.Linear(ch//8, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(2)).unsqueeze(2)

class TemporalAttention(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.a = nn.Linear(h, 1)
    def forward(self, x):
        return (x * torch.softmax(self.a(x), 1)).sum(1)



In [ ]:
class MultiScaleCnnLstm(nn.Module):
    def __init__(self, n_sensors=10, n_classes=6):
        super().__init__()
        def branch(k):
            return nn.Sequential(
                nn.Conv1d(n_sensors, 64, k, padding=k//2),
                nn.BatchNorm1d(64), nn.ReLU(),
                nn.Conv1d(64, 128, k, padding=k//2),
                nn.BatchNorm1d(128), nn.ReLU(),
                nn.MaxPool1d(2), nn.Dropout(0.25))
        self.bs = branch(3)
        self.bm = branch(7)
        self.bl = branch(15)
        self.ca = ChannelAttention(384)
        self.cm = nn.Sequential(
            nn.Conv1d(384, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.MaxPool1d(2), nn.Dropout(0.25),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.MaxPool1d(2), nn.Dropout(0.25))
        self.lstm = nn.LSTM(256, 128, 2, batch_first=True, dropout=0.3, bidirectional=True)
        self.ta   = TemporalAttention(256)
        self.cls  = nn.Sequential(
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_classes))

    def forward(self, x):
        x = x.permute(0,2,1)
        s,m,l = self.bs(x), self.bm(x), self.bl(x)
        t = min(s.size(2), m.size(2), l.size(2))
        x = self.ca(torch.cat([s[:,:,:t],m[:,:,:t],l[:,:,:t]], 1))
        x = self.cm(x).permute(0,2,1)
        x, _ = self.lstm(x)
        return self.cls(self.ta(x))

class SmoothBCE(nn.Module):
    def __init__(self, pw, s=0.05):
        super().__init__()
        self.pw, self.s = pw, s
    def forward(self, logits, y):
        y = y*(1-self.s) + 0.5*self.s
        return nn.functional.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pw)

def train_model(X_tr, y_tr, X_tune, y_tune, epochs=80, batch_size=256):
    X_tr, y_tr = augment_minority(X_tr, y_tr, factor=4)
    print(f"    After augmentation: {len(X_tr):,}")

    model     = MultiScaleCnnLstm(len(ALL_COLS), len(SUPERCLASSES)).to(DEVICE)
    criterion = SmoothBCE(compute_pos_weights(y_tr))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
                    batch_size=batch_size, shuffle=True, num_workers=0)

    best_mcc, best_w, patience = -1, None, 0

    for epoch in range(1, epochs+1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # Evaluate on tune set every 5 epochs
        if epoch % 5 == 0:
            probs = get_probs(model, X_tune)
            preds = (probs > 0.5).astype(int)
            mcc   = calculate_mcc(y_tune, preds, verbose=False)
            if mcc > best_mcc:
                best_mcc = mcc
                best_w   = {k: v.clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
            if epoch % 20 == 0:
                print(f"    Ep {epoch:03d}/{epochs} | tune_MCC {mcc:+.4f} | "
                      f"best {best_mcc:+.4f} | pat {patience}/12")
            if patience >= 12:
                print(f"    Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_w)
    return model

# ── Load ──────────────────────────────────────────────────────────────────────

# ── Leave-One-Out CV ──────────────────────────────────────────────────────────
test_mccs = []

for fold in range(1, 5):
    test_id   = fold
    # ✅ KEY CHANGE: train on ALL 3 non-test experiments
    train_ids = [i for i in range(1, 5) if i != test_id]

    print(f"\n{'='*60}")
    print(f"Fold {fold}/4 | Train={train_ids} Test={test_id}")

    # Concatenate all 3 training experiments
    train_df = pd.concat([exp_dfs[i] for i in train_ids], ignore_index=True)
    test_df  = exp_dfs[test_id]

    X_all, y_all, scaler = make_windows(train_df, fit_scaler=True, stride=STRIDE_TRAIN)
    X_test, y_test, _    = make_windows(test_df,  scaler=scaler,   stride=STRIDE_TEST)

    # ✅ KEY CHANGE: use last 20% of training windows for threshold tuning
    # (time-based split — no future leakage since windows are ordered by time)
    split = int(len(X_all) * 0.80)
    X_tr,   y_tr   = X_all[:split], y_all[:split]
    X_tune, y_tune = X_all[split:], y_all[split:]

    print(f"  Train:{len(X_tr):,} | Tune:{len(X_tune):,} | Test:{len(X_test):,}")

    model = train_model(X_tr, y_tr, X_tune, y_tune)

    # Tune thresholds on the held-out 20%
    tune_probs = get_probs(model, X_tune)
    thresholds = find_best_thresholds(tune_probs, y_tune)

    # Evaluate on test
    test_probs = get_probs(model, X_test)
    y_pred     = (test_probs > thresholds).astype(int)

    print(f"  Per-class MCC (test):")
    mcc = calculate_mcc(y_test, y_pred)
    test_mccs.append(mcc)
    print(f"  Fold MCC: {mcc:+.4f}")
    torch.save(model.state_dict(), f'model_fold{fold}.pt')

print(f"\n{'='*60}")
print(f"Fold MCCs : {[f'{m:+.4f}' for m in test_mccs]}")
print(f"FINAL MCC : {np.mean(test_mccs):+.4f}")